# proceso de entrenamiento y selección de modelos

In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

import joblib
import os

print("Versión de sklearn:", __import__("sklearn").__version__)

Versión de sklearn: 1.6.1


In [2]:
ruta_data = "../data/interim/feature_exploration_scaled.csv"
df = pd.read_csv(ruta_data)

print("Shape del dataframe:", df.shape)
df.head()

Shape del dataframe: (2823, 41)


,ORDERDATE,CITY,PRODUCTLINE,STATUS,QUANTITYORDERED,PRICEEACH,SALES,CITY_TOP,CITY_TOP_Madrid,CITY_TOP_Manchester,...,STATUS_Shipped,SALES_LOG1P,PRICEEACH_LOG1P,QUANTITYORDERED_YJ,SALES_LOG1P_STD,PRICEEACH_LOG1P_STD,QUANTITYORDERED_YJ_STD,SALES_MM,PRICEEACH_MM,QUANTITYORDERED_MM
0,2003-01-06,Nashua,Vintage Cars,Shipped,30.0,100.000,5151.00,Other,0.0,0.0,...,1.0,8.547140,4.615121,10.779233,0.977215,0.744732,-0.502703,0.515174,1.000000,0.279486
1,2003-01-06,Nashua,Vintage Cars,Shipped,50.0,67.800,3390.00,Other,0.0,0.0,...,1.0,8.128880,4.231204,14.925497,0.161327,-0.604671,1.557932,0.302584,0.519159,0.838457
2,2003-01-06,Nashua,Vintage Cars,Shipped,22.0,86.510,1903.22,Other,0.0,0.0,...,1.0,7.551828,4.471753,8.805943,-0.964311,0.240819,-1.483401,0.123099,0.798554,0.055897
3,2003-01-06,Nashua,Vintage Cars,Shipped,49.0,34.470,1689.03,Other,0.0,0.0,...,1.0,7.432502,3.568687,14.736932,-1.197077,-2.933305,1.464218,0.097242,0.021444,0.810509
4,2003-01-09,Frankfurt,Vintage Cars,Shipped,45.0,33.034,1404.00,Other,0.0,0.0,...,1.0,7.247793,3.527360,13.966075,-1.557384,-3.078563,1.081113,0.062833,0.000000,0.698714


# 80/20

In [4]:
TARGET_COL = "SALES"   # ajusta el nombre si tu columna objetivo se llama distinto

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

print("Shape X:", X.shape, " | Shape y:", y.shape)

Shape X: (2823, 40)  | Shape y: (2823,)


In [5]:
n_total = len(df)
n_train = int(n_total * 0.8)

X_train = X.iloc[:n_train].copy()
y_train = y.iloc[:n_train].copy()

X_val = X.iloc[n_train:].copy()
y_val = y.iloc[n_train:].copy()

print("Tamaño train:", X_train.shape, y_train.shape)
print("Tamaño validación:", X_val.shape, y_val.shape)

Tamaño train: (2258, 40) (2258,)
Tamaño validación: (565, 40) (565,)


# pipeline de features

In [6]:
ruta_feature_pipe = "./models/feature_pipeline.pkl"

assert os.path.exists(ruta_feature_pipe), f"No encuentro: {ruta_feature_pipe}"

feature_pipeline = joblib.load(ruta_feature_pipe)
print("Tipo de feature_pipeline:", type(feature_pipeline))

Tipo de feature_pipeline: <class 'sklearn.pipeline.Pipeline'>


# Modelos y configuraciones

In [7]:
model_configs = {
    "linear_regression": {
        "estimator": LinearRegression,
        "params": [
            {"fit_intercept": True,  "positive": False},
            {"fit_intercept": False, "positive": False},
            {"fit_intercept": True,  "positive": True},
        ],
    },
    "ridge": {
        "estimator": Ridge,
        "params": [
            {"alpha": 0.1},
            {"alpha": 1.0},
            {"alpha": 10.0},
        ],
    },
    "random_forest": {
        "estimator": RandomForestRegressor,
        "params": [
            {"n_estimators": 100, "max_depth": None,  "min_samples_split": 2, "random_state": 42},
            {"n_estimators": 200, "max_depth": 10,    "min_samples_split": 2, "random_state": 42},
            {"n_estimators": 300, "max_depth": 20,    "min_samples_split": 5, "random_state": 42},
        ],
    },
    "gradient_boosting": {
        "estimator": GradientBoostingRegressor,
        "params": [
            {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 3, "random_state": 42},
            {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3, "random_state": 42},
            {"n_estimators": 200, "learning_rate": 0.1, "max_depth": 4, "random_state": 42},
        ],
    },
    "knn": {
        "estimator": KNeighborsRegressor,
        "params": [
            {"n_neighbors": 3,  "weights": "uniform"},
            {"n_neighbors": 5,  "weights": "distance"},
            {"n_neighbors": 10, "weights": "distance"},
        ],
    },
}

len(model_configs)

5

# Entrenamiento

In [9]:
resultados = []
best_rmse = np.inf
best_model_name = None
best_config = None
best_pipeline = None

for model_name, cfg in model_configs.items():
    EstimatorClass = cfg["estimator"]
    
    for i, param_dict in enumerate(cfg["params"], start=1):
        print(f"\nEntrenando {model_name} - config {i} con params: {param_dict}")
        
        # Crear instancia del modelo con estos hiperparámetros
        model = EstimatorClass(**param_dict)
        
        # Pipeline completo = features (preprocesamiento) + modelo
        full_pipeline = Pipeline(steps=[
            ("features", feature_pipeline),
            ("model", model),
        ])
        
        # Entrenar en el 80% inicial
        full_pipeline.fit(X_train, y_train)
        
        # Predecir en el 20% final (validación)
        y_pred = full_pipeline.predict(X_val)
        
        # Calcular RMSE
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        print(f"→ RMSE validación: {rmse:0.4f}")
        
        resultados.append({
            "model": model_name,
            "config_id": i,
            "params": param_dict,
            "rmse_val": rmse,
        })
        
        # Actualizar mejor modelo
        if rmse < best_rmse:
            best_rmse = rmse
            best_model_name = model_name
            best_config = param_dict
            best_pipeline = full_pipeline

print("\n=== Mejor modelo encontrado ===")
print("Modelo:", best_model_name)
print("Hiperparámetros:", best_config)
print("RMSE validación:", best_rmse)


Entrenando linear_regression - config 1 con params: {'fit_intercept': True, 'positive': False}
→ RMSE validación: 0.0000

Entrenando linear_regression - config 2 con params: {'fit_intercept': False, 'positive': False}
→ RMSE validación: 350.6202

Entrenando linear_regression - config 3 con params: {'fit_intercept': True, 'positive': True}
→ RMSE validación: 0.0000

Entrenando ridge - config 1 con params: {'alpha': 0.1}
→ RMSE validación: 0.6100

Entrenando ridge - config 2 con params: {'alpha': 1.0}
→ RMSE validación: 5.5733

Entrenando ridge - config 3 con params: {'alpha': 10.0}
→ RMSE validación: 43.5375

Entrenando random_forest - config 1 con params: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'random_state': 42}
→ RMSE validación: 6.6898

Entrenando random_forest - config 2 con params: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 2, 'random_state': 42}
→ RMSE validación: 6.0213

Entrenando random_forest - config 3 con params: {'n_estimators': 

c:\Users\lalvarez\AppData\Local\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\lalvarez\AppData\Local\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\lalvarez\AppData\Local\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\lalvarez\AppData\Local\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(ar

In [11]:
resultados_df = pd.DataFrame(resultados)
resultados_df.sort_values(by="rmse_val", inplace=True)
resultados_df.reset_index(drop=True, inplace=True)
resultados_df

,model,config_id,params,rmse_val
0,linear_regression,3,"{'fit_intercept': True, 'positive': True}",1.169921e-12
1,linear_regression,1,"{'fit_intercept': True, 'positive': False}",4.469992e-12
2,ridge,1,{'alpha': 0.1},6.100347e-01
3,ridge,2,{'alpha': 1.0},5.573334e+00
4,random_forest,2,"{'n_estimators': 200, 'max_depth': 10, 'min_sa...",6.021305e+00
5,random_forest,1,"{'n_estimators': 100, 'max_depth': None, 'min_...",6.689791e+00
6,random_forest,3,"{'n_estimators': 300, 'max_depth': 20, 'min_sa...",7.664740e+00
7,gradient_boosting,2,"{'n_estimators': 200, 'learning_rate': 0.05, '...",8.575506e+00
8,gradient_boosting,3,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",9.326924e+00
9,gradient_boosting,1,"{'n_estimators': 100, 'learning_rate': 0.1, 'm...",1.566995e+01


In [12]:
print("Reentrenando el mejor modelo con todos los datos disponibles...")
best_pipeline.fit(X, y)

# Ruta donde guardaremos el pipeline completo
ruta_full_pipeline = "./models/full_sales_forecast_pipeline.pkl"

joblib.dump(best_pipeline, ruta_full_pipeline)
print("Pipeline completo guardado en:", ruta_full_pipeline)

Reentrenando el mejor modelo con todos los datos disponibles...
Pipeline completo guardado en: ./models/full_sales_forecast_pipeline.pkl


In [13]:
loaded_pipeline = joblib.load(ruta_full_pipeline)

# Tomamos las últimas 5 filas como ejemplo
X_sample = X.tail(5)
y_true_sample = y.tail(5)

y_pred_sample = loaded_pipeline.predict(X_sample)

res_prueba = pd.DataFrame({
    "y_real": y_true_sample.values,
    "y_pred": y_pred_sample,
})
res_prueba

,y_real,y_pred
0,1895.94,1895.94
1,4692.60,4692.60
2,5894.94,5894.94
3,2702.04,2702.04
4,3777.58,3777.58


### Resumen de entrenamiento y selección de modelo

- Se utilizó el dataset `feature_exploration_scaled.csv` con la variable objetivo **SALES**.
- Se realizó una partición **secuencial 80% / 20%** para entrenamiento y validación,
  respetando el orden temporal de las observaciones.
- Se cargó el pipeline de ingeniería de características `feature_pipeline.pkl`,
  que incorpora imputación, codificación, tratamiento de outliers, transformaciones
  y escalado de variables, incluyendo banderas para tienda y producto.
- Se evaluaron **5 modelos de regresión** (LinearRegression, Ridge, RandomForest,
  GradientBoosting y KNN) con **3 configuraciones de hiperparámetros cada uno**
  (15 entrenamientos en total).
- La métrica de comparación fue el **RMSE** sobre el conjunto de validación (20% final).
- El modelo con menor RMSE se seleccionó como **modelo ganador** y se reentrenó
  con el 100% de los datos disponibles.
- Finalmente, se guardó el **pipeline completo (preprocesamiento + modelo ganador)**
  en el archivo `full_sales_forecast_pipeline.pkl`, que será utilizado en
  `05_inference_calculation.ipynb` para generar predicciones sobre nuevos datos.